In [6]:
!pip install pandas
!pip install openpyxl


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd

# Show all rows
pd.set_option("display.max_rows", None)

# Show all columns
pd.set_option("display.max_columns", None)

# Disable column width truncation"
pd.set_option("display.max_colwidth", None)

# Use full notebook width
pd.set_option("display.width", None)

In [4]:
company_name = '360 One Wealth and Asset Management Limited'

In [ ]:
import pandas as pd

# Load data
df = pd.read_excel("20251231 portfolio.xlsx")

# Filter
df = df[df["CPM_NAME"] == "HDFC Securities Ltd"].copy()

# Ensure datetime
df["MATURITY DATE"] = pd.to_datetime(df["MATURITY DATE"])

# Create month bucket
df["Maturity Month"] = df["MATURITY DATE"].dt.to_period("M").dt.to_timestamp()

# Aggregate
pivot = (
    df
    .groupby(["FM_SHORTNAME", "Maturity Month"], as_index=False)[["AUM", "PERCENTAGE_AUM"]]
    .sum()
)

# Sort chronologically
pivot = pivot.sort_values("Maturity Month")

# Convert to display format
pivot["MATURITY DATE"] = pivot["Maturity Month"].dt.strftime("%b %Y")

# Drop helper column
pivot = pivot.drop(columns="Maturity Month")

# Round values
pivot["AUM"] = pivot["AUM"].round(2)
pivot["PERCENTAGE_AUM"] = pivot["PERCENTAGE_AUM"].round(1)

# Reorder columns
pivot = pivot[["FM_SHORTNAME", "MATURITY DATE", "AUM", "PERCENTAGE_AUM"]]

# -----------------------------
# Add Total Row
# -----------------------------
total_row = pd.DataFrame({
    "FM_SHORTNAME": ["Total"],
    "MATURITY DATE": [""],
    "AUM": [pivot["AUM"].sum().round(2)],
    "PERCENTAGE_AUM": [pivot["PERCENTAGE_AUM"].sum().round(1)]
})

pivot = pd.concat([pivot, total_row], ignore_index=True)
pivot["PERCENTAGE_AUM"] = pivot["PERCENTAGE_AUM"].astype(str) + "%"

# Final output
pivot

,FM_SHORTNAME,MATURITY DATE,AUM,PERCENTAGE_AUM
0,LIQUID,Jan 2026,90488.73,0.8%
1,LIQUID,Feb 2026,180977.47,2.2%
2,LIQUID,Mar 2026,90488.73,1.6%
3,Total,,361954.93,4.6%


In [8]:
prompt = """
You are a senior institutional credit analyst at an Asset Management Company (AMC).

Your task is to extract and write a "Company Profile" section for a credit evaluation report.

CRITICAL INSTRUCTIONS:
- Use ONLY the provided document (annual report / company document)
- Do NOT assume or hallucinate any information
- Maintain a formal, factual, and concise tone
- Write in paragraph format (not bullets unless necessary)
- Ensure the output is directly usable in a credit memo

STRUCTURE TO FOLLOW:

1. Ownership & Corporate Structure
- Parent company, shareholding structure, key changes over time
- Subsidiaries / JVs (if material)

2. Business Overview
- Core business activities
- Key revenue streams
- Products/services offered

3. Distribution & Customer Base
- Customer segments (retail, institutional, etc.)
- Scale (clients, AUM, reach, etc.)
- Distribution network (branches, digital platforms)

4. Strategic Positioning
- Market positioning (e.g., leading player, niche, etc.)
- Key strengths (distribution, brand, parent support)

5. Recent Developments
- New subsidiaries, expansions, strategic initiatives
- Regulatory or structural changes

STYLE GUIDE:
- Write in clean, professional paragraphs similar to institutional credit reports
- Use numbers and facts where available
- Avoid repetition
- Avoid promotional language
- Limit output to ~200–300 words
- Prefer dense information over descriptive language

OUTPUT EXAMPLE STYLE:
(Hint: emulate the tone and density of a CRISIL / ICRA / internal credit note)

Now generate the Company Profile based on the provided document.
"""

In [10]:
!pip install google-genai

  Using cached google_genai-2.2.0-py3-none-any.whl.metadata (52 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached google_auth-2.52.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cryptography-48.0.0-cp311-abi3-macosx_10_9_universal2.whl.metadata (4.3 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key="AQ.Ab8RN6I1VBT1XbDqKjH8TDVIXvvDnGglxNNKyqh0NUxAEjEJJg")

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note copy/Angel One Limited-CPR-16102024.pdf"
)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

**Company Profile: Angel One Limited**

**Ownership & Corporate Structure**
Angel One Limited (Angel), incorporated in 1997, was promoted by Mr. Dinesh Thakkar. As of June 30, 2024, the shareholding structure consists of the promoter group (35.6%), individuals/HUF/LLP (35.0%), Foreign Portfolio Investors (15.4%), and mutual funds (11.3%). The company is currently reorganizing its structure to transfer its direct online broking business to a wholly owned subsidiary, Angel Crest Limited (ACL), and its assisted broking services to another subsidiary, Angel Securities Limited (ASL), via a slump sale. This restructuring is intended to streamline operations without impacting the consolidated credit profile.

**Business Overview**
Angel is a leading digital retail broker operating across the equity, derivatives, and commodity segments. The company has transitioned from a traditional firm to a prominent digital-first player, with online channels now contributing approximately 77% of broking income. Its revenue profile is heavily concentrated in the futures and options (F&O) segment, which accounts for 84% of gross broking income. To diversify revenue streams, Angel has recently forayed into wealth management, asset management (awaiting regulatory approval), and credit/fixed income distribution.

**Distribution & Customer Base**
As of June 2024, Angel ranked third in terms of active National Stock Exchange (NSE) clients, catering to ~67 lakh active clients with a 15% market share. The company’s distribution strategy leverages a hybrid model: its direct digital platform accounts for 80% of new acquisitions, while a network of over 10,000 authorized persons supports assisted channels. The customer base is increasingly youthful and geographically diverse; 59% of clients onboarded in FY2024 were from Tier III cities and beyond, and 48% were below the age of 25.

**Strategic Positioning & Recent Developments**
Angel maintains a strong market position characterized by robust profitability, evidenced by a five-year average return on equity (RoE) of 37% (FY2020–FY2024). In Q1 FY2025, the company strengthened its capital base through an equity raise of Rs. 1,500 crore, maintaining a comfortable consolidated financial leverage of 0.5 times as of June 2024. Recent strategic focus has been directed toward the material scale-up of its debt-funded margin trade funding (MTF) book, which reached Rs. 2,915 crore by June 30, 2024, representing a 92% sequential growth.

In [14]:
from google import genai
from google.genai import types

# Initialize client
client = genai.Client(api_key="AQ.Ab8RN6I1VBT1XbDqKjH8TDVIXvvDnGglxNNKyqh0NUxAEjEJJg")

# 2. Credit Rating Prompt
prompt = f"""
You are a senior institutional credit analyst.

Your task is to extract and write the "Credit Rating" section for {company_name}.

CRITICAL INSTRUCTIONS:
- Focus ONLY on {company_name}
- Do NOT include ratings of parent company, subsidiaries, or peers
- If ratings for {company_name} are NOT available, explicitly state:
  "No publicly available credit rating information found for {company_name}"
- Do NOT infer or approximate ratings

DATA SOURCES:
- Use the provided document as the PRIMARY source
- Use Google Search ONLY to verify or supplement ratings specifically for {company_name}

INCLUDE (only if available for {company_name}):
1. Long-term rating (with outlook if available)
2. Short-term rating
3. Rating agencies (CRISIL, ICRA, CARE, etc.)
4. Interpretation of rating (credit quality)
5. Key drivers (e.g., parent support, capitalization, business strength)

STRICT FILTERING RULE:
- If a rating belongs to:
  - Parent company
  - Group entity
  - Subsidiary
  - Similarly named entity
  → IGNORE it

STYLE:
- Paragraph format
- 100–150 words
- Crisp, factual, no fluff
- Suitable for institutional credit memo

OUTPUT REQUIREMENT:
- Either a clean rating paragraph
- OR a clear statement that rating is not available

Now generate the Credit Rating section for {company_name}.
"""

# 3. Generate response with Google Search grounding
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        prompt
    ],
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )
)

# 4. Output
Markdown(response.text)

360 ONE WAM Limited (formerly IIFL Wealth Management Limited), the parent entity of the 360 ONE group, maintains high-grade credit ratings from major domestic agencies. **ICRA** assigns a long-term rating of **[ICRA]AA (Stable)** to the company’s Non-Convertible Debenture (NCD) and Principal Protected Market-Linked Debenture (PP-MLD) programs. Its short-term debt instruments, including the Commercial Paper (CP) program, are rated at the highest category of **A1+** by **CRISIL**, **ICRA**, and **CARE Ratings**.

These ratings reflect the company’s leading market position as one of India's largest non-bank wealth management players, characterized by comfortable capitalization, strong profitability, and an experienced management team. The ratings are further supported by a transition toward high-yield, annual recurring revenue (ARR) assets and significant operational synergies across its wealth and asset management verticals. Key credit constraints include exposure to capital market fluctuations, regulatory risks inherent in the wealth management industry, and potential attrition of key personnel.

In [15]:
prompt = f"""
You are a senior institutional credit analyst.

Your task is to write a concise "ESG Profile" paragraph for {company_name}, similar to how rating agencies (CRISIL, ICRA, CARE) present ESG commentary in rating rationales.

CRITICAL INSTRUCTIONS:
- Use ONLY the provided document
- Do NOT hallucinate or assume ESG practices
- Maintain strict factual accuracy
- Write in a single, well-structured paragraph (no bullets)

CONTENT GUIDELINES:

1. Environmental (E)
- Assess whether the business has exposure to environmentally sensitive sectors
- If not applicable (e.g., service-oriented business), explicitly state that
- If no environmental disclosures exist, mention the absence implicitly (do not overemphasize)

2. Social (S)
- Include relevant aspects ONLY if available:
  - Nature of customers / business model risk
  - Any responsibility toward clients (if mentioned)
- Do NOT include CSR activities unless very material

3. Governance (G) — MOST IMPORTANT
- Board composition (total directors, independent directors)
- Presence of committees and independence
- Any diversity (e.g., female director)
- Parent/group linkage and governance standards
- Management track record (if mentioned or can be inferred conservatively)

4. OVERALL ESG VIEW
- Provide a concluding line:
  - "ESG profile can be considered Strong / Adequate / Satisfactory / Weak"
- Base this ONLY on available disclosures (do not overstate)

STRICT RULES:
- If ESG disclosures are limited:
  → Do NOT say "no ESG"
  → Instead, infer conservatively from business nature + governance
- Do NOT confuse CSR with ESG
- Avoid generic statements like "committed to sustainability"
- Keep tone neutral and credit-focused

STYLE:
- Single paragraph
- Dense, factual, rating-note style
- No fluff, no headings
- ~120–180 words

OUTPUT SHOULD RESEMBLE:
- CRISIL / ICRA rating rationale ESG paragraph
- Tight, analytical, understated

Now generate the ESG profile for {company_name}.
"""

In [16]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key="AQ.Ab8RN6I1VBT1XbDqKjH8TDVIXvvDnGglxNNKyqh0NUxAEjEJJg")

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf"
)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

FileNotFoundError: /Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf is not a valid file path.

In [10]:
prompt = f"""
You are a senior institutional credit analyst.

Your task is to write the "Key Risks and Mitigants" section for {company_name}, in the exact style of rating agency reports (CRISIL / ICRA / CARE).

CRITICAL INSTRUCTIONS:
- Use ONLY the provided document
- Do NOT hallucinate or add external assumptions
- Maintain strict factual accuracy

WRITING STYLE (VERY IMPORTANT):

1. Structure:
- Each key risk should start with a short heading sentence (one line)
- Followed by a detailed paragraph

2. Flow within each paragraph:
- Start by explaining the risk
- Gradually incorporate mitigants within the same paragraph
- Do NOT explicitly separate "risk" and "mitigant"
- Use natural transitions like:
  "Further,"
  "Additionally,"
  "The company has implemented…"
  "It benefits from…"

3. Tone:
- Neutral, descriptive, and analytical
- Avoid strong judgments (no "high risk", "low risk")
- Use phrases like:
  "susceptible to"
  "dependent on"
  "exposed to"
  "has implemented"
  "benefits from"

4. Content to cover (if available):
- Market/industry volatility
- Regulatory exposure
- Revenue concentration
- Competitive intensity
- Risk management systems
- Parent/group support

STRICT RULES:
- Do NOT use bullet points
- Do NOT force "However" for every mitigant
- Do NOT exaggerate risks
- Do NOT omit mitigants if present in text
- Avoid repetition

OUTPUT FORMAT:

<Heading – Risk 1>
<Paragraph explaining risk and mitigants>

<Heading – Risk 2>
<Paragraph explaining risk and mitigants>

(2–4 such sections)

Keep the output concise, dense, and rating-note ready (~200–300 words total).

Now generate the "Key Risks and Mitigants" section for {company_name}.
"""

In [12]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf"
)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

**Susceptibility to market volatility and its impact on assets under management (AUM)**
The company’s results of operations are materially dependent on its AUM, making its revenue streams vulnerable to adverse movements in equity, debt, and alternative markets. Any event impacting investment performance could potentially reduce the attractiveness of offerings, leading to lower inflows or redemption pressures. To manage these risks, the group has implemented a multi-tiered risk management structure, including a Board-level Risk Management Committee and dedicated Investment and Valuation Committees that ensure assets are managed within approved mandates. Further, the company employs a specialized product team and a Product Approval Committee to shortlist complex offerings and conducts detailed risk appetite assessments for clients to ensure appropriate product placement.

**Exposure to a highly competitive landscape and evolving distribution models**
360 ONE faces significant competition from established domestic and multi-national firms, as well as emerging technology-driven 'Robo-advisors' that offer low-cost alternatives in the HNI and mid-market segments. Additionally, the industry's gradual transition from a commission-based to a fee-based model represents a structural shift that could impact traditional revenue streams. The group addresses these challenges by leveraging its established brand image, extensive industry knowledge, and deep-rooted client relationships. It has also launched "360 ONE Open" and its Advisory platform to specifically attract clients who prefer fee-based engagement. Further, the company has invested in a dedicated technology team to enhance digital engagement and recently acquired ET Money to expand its reach into the mass affluent segment.

**Vulnerability to regulatory changes and operational complexities**
Operating in the strictly regulated financial services sector, the group is subject to supervision by multiple authorities, including SEBI and RBI, where frequent modifications to rules can result in increased compliance costs and management time. Furthermore, the complexity of its operations exposes the group to potential system failures, cyber-attacks, or manual errors. To mitigate these risks, the group maintains a dedicated compliance function and an Internal Audit team that verifies the efficacy of written procedures and maker-checker controls. Additionally, the company has deployed robust information security defences, including firewalls and Intrusion Prevention Systems, overseen by a Chief Information Security Officer. Operational resilience is further supported by a comprehensive Business Continuity and Disaster Recovery plan.

In [13]:
prompt = f"""
You are a senior institutional credit analyst.

Your task is to prepare the "Financial Performance" section for {company_name}, in rating agency style (CRISIL / ICRA / CARE), by interpreting and cleaning the provided data.

---

CRITICAL THINKING INSTRUCTION:

The input data may be messy, broken, or scattered.
You must reconstruct it like a human analyst.

---

PERIOD HANDLING (VERY IMPORTANT):

1. Identify all periods mentioned in the document
   (e.g., Mar-22, Mar-23, Mar-24, Mar-25, Dec-25)

2. For each table:
   - Include ONLY those periods where at least one data point is available
   - Do NOT create rows with completely missing values

3. If a specific metric is missing for a year:
   - Leave that cell blank or use "-"
   - But include the row if other data exists

4. Do NOT force uniform availability across all tables

---

OUTPUT STRUCTURE:

Financial Performance

1. Capital

- Create ONE clean consolidated table:
  Period | Months | Net worth | Borrowings | Net worth/Assets | Debt/Net worth

- Clean formatting issues:
  - "1 2" → 12
  - Broken numbers → fix

- Include only valid periods (no empty rows)

- Then write a SHORT paragraph:
  - Net worth trend
  - Borrowings / gearing trend
  - Key drivers

---

2. Asset Quality

- Create ONE clean table:
  Period | Total Assets | Loans & Advances | Loans/Assets | Gross NPA | Net NPA

- Fix spacing issues:
  - "3 3 . 2 %" → 33.2%

- Include only relevant periods

- Then write a SHORT paragraph:
  - Loan book composition (MTF)
  - Movement in loan book
  - Regulatory impact

---

3. Earnings

- Create ONE clean table:
  Period | NOI | PAT | Cost/Income | Provision/Assets | PAT/Assets | PAT/Net worth

- Fix broken values:
  - "1 8 . 2 0" → 18.20

- Include only periods where data exists

- Then write a SHORT paragraph:
  - Income trend
  - Profitability
  - Cost pressures
  - Regulatory impact

---

4. Liquidity

- No table required

- Write a SHORT paragraph:
  - Borrowings (CPs)
  - Cash & equivalents
  - Credit lines
  - MTF book liquidity

- End with:
  "liquidity can be considered adequate"

---

5. Management

- Short paragraph:
  - Experience
  - Parent linkage
  - Leadership

---

STYLE INSTRUCTIONS:

- Neutral, rating-agency tone
- No adjectives like "strong", "robust"
- Use:
  "increased to"
  "declined to"
  "was impacted by"
  "supported by"

---

STRICT RULES:

- Do NOT include empty rows
- Do NOT hallucinate values
- Do NOT copy messy tables directly
- Do NOT force all years in every table
- Ensure tables are clean and readable

---

SELF-CHECK:

- No row should be completely empty
- All numbers should be clean and properly formatted
- Tables should reflect only available data

---

Now generate the "Financial Performance" section for {company_name}.
"""

In [15]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf"
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

### Financial Performance

**1. Capital**

| Period | Net worth (₹ Cr) | Borrowings (₹ Cr) | Net worth/Assets (%) | Debt/Net worth (x) |
| :----- | :---------------- | :---------------- | :------------------ | :----------------- |
| Mar-25 | 7,025.81          | 11,094.74         | 35.54               | 1.57               |
| Mar-24 | 3,413.84          | 9,411.06          | 22.58               | 2.73               |

The company's net worth increased to ₹7,025.81 Cr in March 2025 from ₹3,413.84 Cr in March 2024. This change reflects an increase in equity through Qualified Institutional Placement (QIP) and other equity movements. Borrowings increased to ₹11,094.74 Cr in March 2025 from ₹9,411.06 Cr in March 2024. The debt/net worth ratio declined to 1.57 times in March 2025 from 2.73 times in March 2024. This decline was impacted by an increase in the group's net worth.

**2. Asset Quality**

| Period | Total Assets (₹ Cr) | Loans & Advances (₹ Cr) | Loans/Assets (%) | Gross NPA (₹ Cr) | Net NPA (₹ Cr) |
| :----- | :------------------ | :---------------------- | :--------------- | :-------------- | :------------ |
| Mar-25 | 19,768.70          | 8,397.37                | 42.48            | 29.44           | -             |
| Mar-24 | 15,118.85          | 6,368.67                | 42.12            | 21.52           | -             |

The company's total assets increased to ₹19,768.70 Cr in March 2025 from ₹15,118.85 Cr in March 2024. Loans and advances increased to ₹8,397.37 Cr in March 2025 from ₹6,368.67 Cr in March 2024. Loans and advances constituted 42.48% of total assets in March 2025, a slight increase from 42.12% in March 2024. The gross non-performing assets (Gross NPA), represented by loss allowance, increased to ₹29.44 Cr in March 2025 from ₹21.52 Cr in March 2024. This movement was impacted by provisions on loans originated during the year and changes in continuing loans.

**3. Earnings**

| Period | NOI (₹ Cr) | PAT (₹ Cr) | Cost/Income (%) | Provision/Assets (%) | PAT/Assets (%) | PAT/Net worth (%) |
| :----- | :--------- | :--------- | :-------------- | :------------------- | :------------- | :---------------- |
| Mar-25 | 2,446.00   | 1,015.30   | 45.9            | 0.056                | 5.13           | 14.45             |
| Mar-24 | 1,846.00   | 804.21     | 48.7            | 0.022                | 5.32           | 23.56             |

The company's Net Operating Income (NOI) increased to ₹2,446.00 Cr in March 2025 from ₹1,846.00 Cr in March 2024. Profit After Tax (PAT) increased to ₹1,015.30 Cr in March 2025 from ₹804.21 Cr in March 2024. The cost to income ratio declined to 45.9% in March 2025 from 48.7% in March 2024. Provision to assets increased to 0.056% in March 2025 from 0.022% in March 2024. PAT as a percentage of assets declined to 5.13% in March 2025 from 5.32% in March 2024. PAT as a percentage of net worth declined to 14.45% in March 2025 from 23.56% in March 2024. The decline in PAT/Net worth was primarily impacted by dilution in equity through QIP.

**4. Liquidity**

The company's commercial paper outstanding increased to ₹936.00 Cr in March 2025 from ₹624.00 Cr in March 2024. Cash and cash equivalents increased to ₹740.19 Cr in March 2025 from ₹442.74 Cr in March 2024. Undrawn backup lines declined to ₹150.00 Cr in March 2025 from ₹340.00 Cr in March 2024. The company utilizes various short-term borrowing facilities and maintains sufficient liquid assets to manage its obligations. The loans from lending operations are subject to interest rate resets based on the Prime Lending Rate (PLR). As such, liquidity can be considered adequate.

**5. Management**

The company's management team has experience in wealth and asset management and financial services. The Board of Directors provides strategic guidance for the company's operations. Key executives have experience in areas such as telecom, financial services, and investment. The company has implemented policies and frameworks to ensure ethical conduct, risk management, and compliance with regulatory requirements.

In [16]:
prompt = f"""
You are a senior institutional credit analyst.

Your task is to write the "Capital" section for {company_name}.

---

STEP 1: DRIVER EXTRACTION (MANDATORY)

Before writing the paragraph, identify and extract ALL relevant explanatory points from the text, such as:

- Reasons for change in net worth (e.g., profit retention, equity infusion)
- Nature of borrowings (e.g., commercial papers)
- Purpose of borrowings (e.g., MTF funding, exchange margins)
- Reasons for change in gearing

IMPORTANT:
- Use ONLY what is explicitly mentioned in the text
- Do NOT infer or generalize
- Do NOT skip qualitative explanations

---

STEP 2: TABLE RECONSTRUCTION (VERY IMPORTANT):

1. Identify all available periods (e.g., Mar-22, Mar-23, Mar-24, Mar-25, Dec-25)

2. Reconstruct ONE clean table with columns:
   Period | Months | Net worth | Borrowings | Net worth/Assets % | Debt/Net worth ratio

   - Do NOT change column names
   - Do NOT change column order
   - Do NOT add or remove columns

3. Fix formatting issues:
   - "1 2" → 12
   - Broken numbers (e.g., "20.29 95.33") → split correctly
   - Ensure proper alignment of values

4. Include ONLY periods where at least one data point is available
   - Do NOT create empty rows

5. Maintain chronological order

---

STEP 3: PARAGRAPH GENERATION (CRITICAL)

Write ONE paragraph that:

1. Starts with:
   → Net worth trend

2. Then:
   → Borrowings structure and purpose

3. Then:
   → Gearing movement

4. MUST incorporate extracted drivers from STEP 1

---

WRITING STYLE (VERY IMPORTANT):

- Use factual, source-aligned language
- Prefer:
  "has been increasing due to..."
  "are primarily in the form of..."
  "used for funding..."
  "increase in gearing was due to..."

- DO NOT:
  - Generalize (e.g., "due to business growth")
  - Add new interpretations
  - Skip specific phrases from text

---

STRICT RULE:

If the text explicitly mentions a driver (e.g., equity infusion, CP borrowings, MTF),
→ it MUST appear in the paragraph

Failure to include source drivers = incorrect output

---

OUTPUT FORMAT:

Capital

<Table>

<Paragraph grounded in extracted drivers>

"""

In [17]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf"
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

Capital

| Period | Months | Net worth | Borrowings | Net worth/Assets % | Debt/Net worth ratio |
|:-------|:-------|:----------|:-----------|:-------------------|:---------------------|
| Mar-23 | 12     | 3,086.34  | n/a        | n/a                | n/a                  |
| Mar-24 | 12     | 3,413.84  | 9,411.06   | 22.58%             | 2.73                 |
| Mar-25 | 12     | 7,025.81  | 11,094.74  | 35.54%             | 1.57                 |

The Group’s net worth showed a strong upward trend, increasing from ₹3,086.34 Crore at March 31, 2023, to ₹3,413.84 Crore at March 31, 2024, and further to ₹7,025.81 Crore at March 31, 2025. This increase was driven by factors such as profit for the year and a significant equity infusion, including that from a Qualified Institutional Placement (QIP) in October 2024 and issuance of equity shares for consideration other than cash. The tangible return on equity (RoE) excluding goodwill and intangibles decreased to 24.3% in FY25 from 30.1% in FY24, primarily due to the dilution in equity through the QIP. The Group's borrowings, comprising debt securities, other borrowings (other than debt securities), and subordinated liabilities, increased from ₹9,411.06 Crore at March 31, 2024, to ₹11,094.74 Crore at March 31, 2025. Debt securities include secured non-convertible debentures and unsecured commercial papers, with the former secured by a pari passu mortgage and charge over the Company’s identified immovable property and receivables. Other borrowings include term loans from banks, secured by a pari passu charge on specific receivables and backed by a Corporate Guarantee, and collateralized borrowing and lending obligations (CBLO), which are secured against Investments in Government Securities. One of the subsidiaries utilized money from non-convertible debentures for the purposes for which they were raised. The funds raised by way of QIP were largely deployed for the growth of the Group's lending and Alternates’ businesses. Reflecting this substantial increase in net worth, the Debt/Net worth ratio decreased from 2.73 at March 31, 2024, to 1.57 at March 31, 2025. Concurrently, the Net worth/Assets percentage improved from 22.58% at March 31, 2024, to 35.54% at March 31, 2025.

In [18]:
prompt = f"""
You are a senior institutional credit analyst.

Your task is to write the "Asset Quality" section for {company_name}.

---

STEP 1: DRIVER EXTRACTION (MANDATORY)

Before writing the paragraph, identify and extract ALL relevant explanatory points from the text, such as:

- Composition of loans and advances (e.g., MTF book)
- Reasons for movement in the loan book
- Regulatory factors affecting the loan book
- Any change in market conditions affecting asset quality
- Composition of investments
- Any trading securities held
- Any mention of Gross NPA / Net NPA

IMPORTANT:
- Use ONLY what is explicitly mentioned in the text
- Do NOT infer or generalize
- Do NOT skip qualitative explanations

---

STEP 2: TABLE RECONSTRUCTION (VERY IMPORTANT)

- Construct ONE clean consolidated table using EXACTLY the following columns:

  Period | Months | Total Assets | Loans & Advances | Loans/Assets % | Gross NPA | Net NPA

- Do NOT change column names
- Do NOT change column order
- Do NOT add or remove columns

- Identify all available periods (e.g., Mar-22, Mar-23, Mar-24, Mar-25, Dec-25)

- Include ONLY periods where at least one data point is available
- Do NOT create empty rows

- Fix formatting issues:
  - "1 2" → 12
  - "3 3 . 2 %" → 33.2%
  - Broken or misaligned values → place correctly under the appropriate column

- Maintain chronological order

- If a specific metric is unavailable for a period, leave that cell as "-"
- Do NOT hallucinate values

---

STEP 3: PARAGRAPH GENERATION (CRITICAL)

Write ONE paragraph that:

1. Starts with:
   → Composition/nature of loans and advances or asset base

2. Then:
   → Movement in the loan book across the latest periods

3. Then:
   → Specific reasons for the change (must include regulatory and market drivers if mentioned)

4. Then:
   → Other relevant asset-side commentary, such as:
      - investment composition
      - trading securities
      - any asset-quality observations

5. MUST incorporate extracted drivers from STEP 1

---

WRITING STYLE (VERY IMPORTANT):

- Use factual, source-aligned language
- Prefer:
  "Loans and advances comprised..."
  "The loan book decreased due to..."
  "However, during..."
  "Investments comprised..."
  "As on..."
  "The securities are in the form of..."

- DO NOT:
  - Generalize (e.g., "asset quality remained stable" unless explicitly supported)
  - Add new interpretations not in source
  - Use words like "strong", "robust", "healthy"
  - Skip specific phrases from text

---

STRICT RULE:

If the text explicitly mentions a driver (e.g., MTF book, reduced trading volumes, upfront collection of premium, flat fee structure, removal of cross margin benefit, improved capital market sentiments, investment mix),
→ it MUST appear in the paragraph.

Failure to include source drivers = incorrect output

---

OUTPUT FORMAT:

Asset Quality

<Clean reconstructed table with exact columns>

<Paragraph grounded in extracted drivers>
"""

In [20]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf"
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

Asset Quality

| Period | Months | Total Assets | Loans & Advances | Loans/Assets % | Gross NPA | Net NPA |
|:---------------|:-------|:-------------|:-----------------|:---------------|:----------|:--------|
| March 31, 2024 | -      | 15,118.85    | 6,368.67         | 42.12%         | -         | -       |
| March 31, 2025 | -      | 19,768.70    | 8,397.37         | 42.48%         | -         | -       |

The Group's loans and advances primarily comprise loans to staff and inter-corporate deposits, mainly extended to Ultra High Net Worth Individuals (UHNIs) against securities/collaterals (Loans against Securities - LAS) and to related parties for general business purposes. As of March 31, 2025, the total loan book increased to ₹8,397.37 crore from ₹6,368.67 crore as of March 31, 2024. For FY 2024-25, this movement is reflected in the provision on loans originated during the year of ₹15.45 crore, a net change in provision on continuing loans of ₹1.65 crore, and provisions on loans derecognised of (₹9.18) crore. The Group operates within a highly regulated financial services sector, with its activities subject to supervision and regulation by multiple statutory and regulatory authorities including SEBI and RBI. Capital level and requirements for the loan book are assessed against long- and short-term strategies and regulatory requirements. An Asset Liability Management (ALM) Framework is in place to regularly monitor and manage maturity profiles of financial assets and financial liabilities, including debt financing plans. The Group manages market risk, specifically other price risk, which relates to changes in the market reference price of fair-valued investments, derivative financial instruments, and debt securities. For loans, credit risk is considered insignificant, and no Expected Credit Loss (ECL) provision is made for loans given to related parties and employees. For trade receivables, a lifetime Expected Credit Loss (ECL) provision is made based on a provision matrix and historical experience. The measurement of impairment losses across all categories of financial assets requires judgment and estimates based on future cash flows and collateral values. Credit risk on bank balances, deposits, and other financial assets is limited due to high credit ratings and capital adequacy. No explicit Gross NPA or Net NPA values are provided in the financial statements. Investments comprise mutual funds, debt securities, government securities, equity instruments, alternate investment funds, and other securities. These investments are also kept as collateral against borrowings. The trading portfolio includes investments and derivatives, which are measured at fair value through profit or loss. The Company’s financial accounting and reporting processes are highly dependent on IT systems, with the risk of material misstatement due to IT control environment gaps identified as a key audit matter. Contingent liabilities include corporate guarantees, disputed income tax demands, legal matters, and indirect tax disputes, for which management believes the ultimate outcome will not have a material adverse impact on the financial position, results of operations, and cash flows. A civil suit settlement of GBP 11.10 million (₹87.63 crore net of taxes) was recorded as an exceptional item in the Standalone Statement of Profit and Loss. A search by the Income Tax Department for FY25 is underway, with the ultimate outcome uncertain, but no adjustments or disclosures identified as of the financial statement approval date.

In [21]:
prompt = f"""
You are a senior institutional credit analyst.

Your task is to write the "Management" section for {company_name}, in rating agency style (CRISIL / ICRA / CARE).

---

STEP 1: DRIVER EXTRACTION (MANDATORY)

Before writing, extract ALL relevant management-related information from the text, such as:

- Years of operation / track record
- Experience of key personnel (especially CEO/MD)
- Background of key management (e.g., prior roles, institutions)
- Linkage with parent/group (e.g., HDFC Bank)
- Board representation (e.g., nominee directors)
- Tenure and stability of management team

IMPORTANT:
- Use ONLY what is explicitly mentioned
- Do NOT infer or generalize
- Do NOT skip any relevant qualitative detail

---

STEP 2: PARAGRAPH GENERATION (CRITICAL)

Write ONE concise paragraph that follows this flow:

1. Start with:
   → Overall experience / track record of the company

2. Then:
   → Leadership profile (MD/CEO and key roles)

3. Then:
   → Parent linkage and board representation

4. Then:
   → Management stability / tenure

---

MUST INCLUDE (if present in text):

- Years of operation (e.g., over two decades)
- MD & CEO background (Mr. Dhiraj Relli)
- Prior experience at HDFC Bank
- Board linkage (e.g., nominee directors like Arvind Vohra)
- Presence of HDFC Bank personnel in key roles

---

WRITING STYLE (VERY IMPORTANT):

- Neutral, factual, and concise
- Use phrases like:
  "has been operating for over..."
  "is led by an experienced management team"
  "the MD & CEO has been associated with..."
  "has previously served as..."
  "further, ... serves as nominee director"
  "several personnel have been tenured employees of..."

- DO NOT:
  - Use words like "strong", "excellent", "highly experienced"
  - Add new interpretation
  - Over-explain

---

STRICT RULE:

All explicitly mentioned management details MUST be included.

Missing:
- CEO background
- parent linkage
- board representation
→ makes output incomplete

---

OUTPUT FORMAT:

Management

<Single analytical paragraph>
"""

In [22]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf"
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

**Management**

360 ONE WAM Limited, incorporated in January 2008, has been operating for over 17 years and was successfully listed on BSE and NSE in 2019. The management team is led by Mr. Karan Bhagat, Managing Director and Founder, who has been re-appointed for a five-year term from July 27, 2025, to July 26, 2030. Mr. Sanjay Wadhwa serves as the Chief Financial Officer, and Mr. Rohit Bhase as the Company Secretary and Compliance Officer; Mr. Pankaj Fitkariwala was included as Senior Management (COO) with effect from July 17, 2025. The Board includes Mr. Akhil Gupta as an Independent Director and Chairman. Nominee Directors, Mr. Rishi Mandawat and Mr. Pavninder Singh, are nominated by BC Asia Investments X Limited, an equity shareholder. Mr. Saahil Murarka, who has over 12 years of diverse experience and currently serves as Managing Director at Batlivala & Karani Securities, a wholly owned subsidiary of the Company, joined the Board as an Additional, Non-Executive, Non-Independent Director with effect from July 17, 2025. The company has engaged in strategic collaborations, including an exclusive partnership with UBS AG and acquisitions such as B&K Securities and ET Money. The average tenure of Directors on the Board is 4.56 years, and the company maintains a regrettable attrition rate of 5.3% for its 1,207 permanent employees as of FY25.

In [23]:
prompt = f"""
You are a senior institutional credit analyst.

Your task is to write the "Earnings" section for {company_name}.

---

STEP 1: DRIVER EXTRACTION (MANDATORY)

Before writing the paragraph, identify and extract ALL relevant explanatory points from the text, such as:

- Drivers of income growth (e.g., interest income, broking income)
- Regulatory impacts (e.g., F&O regulations, "true to label", account charge changes)
- Market factors (e.g., equity market weakness, trading volumes)
- Revenue mix (e.g., interest income %, broking income %)
- Strategy changes (e.g., focus on non-brokerage income, distribution, MTF)
- Cost drivers:
  - interest expenses (e.g., CP borrowings)
  - operating expenses (e.g., advertising, customer acquisition, trading costs)
- Profitability impact (e.g., moderation in returns)

IMPORTANT:
- Use ONLY what is explicitly mentioned in the text
- Do NOT infer or generalize
- Do NOT skip qualitative explanations

---

STEP 2: TABLE RECONSTRUCTION (VERY IMPORTANT)

- Construct ONE clean consolidated table using EXACTLY the following columns:

  Period | Months | NOI | PAT | Cost/Income | Provision/Assets | PAT/Assets | PAT/Net worth

- Do NOT change column names
- Do NOT change column order
- Do NOT add or remove columns

- Identify all available periods (e.g., Mar-22, Mar-23, Mar-24, Mar-25, Dec-25)

- Include ONLY periods where at least one data point is available
- Do NOT create empty rows

- Fix formatting issues:
  - "1 8 . 2 0" → 18.20
  - "0 . 1 %" → 0.1%
  - "C o s t / i n c o m e" → Cost/Income
  - Misaligned values → correct placement

- Maintain chronological order

- If a value is missing → use "-"
- Do NOT hallucinate values

---

STEP 3: PARAGRAPH GENERATION (CRITICAL)

Write ONE paragraph that follows this flow:

1. Start with:
   → Income / NOI trend

2. Then:
   → Key drivers of income (interest income, broking income)

3. Then:
   → Regulatory and market impact (VERY IMPORTANT)

4. Then:
   → Cost structure:
      - interest expenses
      - operating expenses

5. Then:
   → Profitability / return metrics trend

6. Then:
   → Revenue mix and strategic shift (non-brokerage, MTF, distribution)

---

MUST INCLUDE:

- Growth in NOI and its drivers
- Impact of F&O regulations and market weakness
- Impact of regulatory changes in 9MFY26
- Increase in interest expenses due to CP borrowings
- Increase in operating expenses (advertising, acquisition, trading costs)
- Moderation in return ratios
- Revenue mix (~47% interest, ~49% broking)

---

WRITING STYLE (VERY IMPORTANT):

- Use factual, source-aligned language
- Prefer:
  "NOI increased due to..."
  "However, growth was impacted by..."
  "Operating income was affected due to..."
  "Interest expenses increased due to..."
  "Operating expenses remained elevated due to..."
  "Consequently, return ratios moderated..."

- DO NOT:
  - Use words like "strong", "robust", "healthy"
  - Add new interpretations
  - Skip regulatory or cost drivers
  - Over-explain

---

STRICT RULE:

If a driver is explicitly mentioned (regulation, cost increase, revenue mix, strategy shift),
→ it MUST be included in the paragraph.

Failure to include source drivers = incorrect output

---

OUTPUT FORMAT:

Earnings

<Clean reconstructed table with exact columns>

<Single analytical paragraph grounded in extracted drivers>
"""

In [24]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf"
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

Earnings

| Period | Months | NOI | PAT | Cost/Income | Provision/Assets | PAT/Assets | PAT/Net worth |
|---|---|---|---|---|---|---|---|
| FY21 | 12 | 1,053 | 369 | 54% | - | - | 15.0% |
| FY22 | 12 | 1,535 | 582 | 51% | - | - | 24.7% |
| FY23 | 12 | 1,569 | 668 | 46% | - | - | 26.7% |
| FY24 | 12 | 1,965 | 802 | 49% | 0.34% | 5.32% | 30.1% |
| FY25 | 12 | 2,652 | 1,015 | 46% | 0.35% | 5.14% | 24.3% |

Total Revenues (NOI) in FY25 grew by 35.0% year-on-year to ₹2,652 Cr, with Profit After Tax (PAT) increasing by 26.6% to ₹1,015 Cr. This income growth was supported by overall ARR AUM rising to ₹2,46,828 Cr, up 23% year-on-year, driven by net flows of ₹25,974 Cr. Wealth ARR net flows increased by 42% year-on-year to ₹22,334 Cr, while Asset gross flows reached approximately ₹16,300 Cr. Key drivers of income included Annual Recurring Revenues (ARR), which grew by 28.2% year-on-year to ₹1,701 Cr, along with higher transactional/brokerage income primarily from opportunities in the capital markets. Consolidated interest income increased to ₹1,063.89 Cr in FY25 from ₹766.36 Cr in FY24, and consolidated transactional/brokerage revenue increased to ₹744 Cr from ₹519 Cr. The period was characterized by a volatile global environment, market peaks, intermittent corrections, and geopolitical flux. FII flows were volatile in FY25, resulting in a net outflow of ₹1.27 Trn, influenced by geopolitical tensions, Fed rate uncertainties, and a stronger dollar. In contrast, Domestic Institutional Investors (DIIs) recorded inflows of ₹6.06 Trn, driven by SIP participation and retail enthusiasm. The RBI projected a real GDP growth of 6.5% for FY26 and CPI inflation around 4.0%. The broking industry in India experienced growth, fueled by increasing investor participation and access to capital markets, with demat accounts reaching 192.4 Mn in FY25 and NSE average daily turnover in equity cash segment growing 38% to ₹1.12 Trn. The low penetration of mutual funds in India, with less than 5% of the population actively participating, suggests significant growth potential. The NBFC lending market showed credit expansion and growth potential. Management noted the ongoing need for tighter control and more transparency in regulations and policies, which may impact business operations, products, pricing, and technological processes. Total Costs increased by 27.3% to ₹1,218 Cr. Employee costs increased by 28.6% year-on-year to ₹912 Cr, with fixed employee costs at ₹587 Cr and variable costs at ₹325 Cr. Admin and Other expenses increased by 23.5% year-on-year to ₹306 Cr. Finance cost increased, influenced by a rise in interest on borrowings to ₹216.48 Cr in FY25 from ₹149.70 Cr in FY24 and interest on debt securities to ₹648.92 Cr from ₹478.56 Cr, with commercial paper borrowings contributing to finance costs. The Cost-to-Income ratio was 45.9% in FY25, a decrease from 48.7% in FY24. The Return on Equity (excluding Goodwill & Intangibles) decreased to 24.3% in FY25 from 30.1% in FY24, attributed to dilution in equity through a Qualified Institutional Placement (QIP). Net profit margin stood at 38.3% in FY25 compared to 40.8% in FY24. ARR revenues contributed 70% to total revenues from operations, with transactional/brokerage income also contributing significantly. Strategic acquisitions of B&K Securities and ET Money, along with the collaboration with UBS AG, aimed to strengthen institutional capabilities, expand digital footprint, and unlock global synergies. The strategic focus included retaining leadership in Wealth Management and Alternatives while expanding into Global and Capital Markets segments, with equity broking and investment banking identified as key drivers in the Capital Markets vertical.

In [25]:
prompt = f"""
You are a senior institutional credit analyst.

Your task is to write the "Liquidity" section for {company_name}, in rating agency style (CRISIL / ICRA / CARE).

---

STEP 1: DRIVER EXTRACTION (MANDATORY)

Before writing, extract ALL relevant liquidity-related points from the text, such as:

- Sources of liquidity:
  - cash and cash equivalents
  - bank balances
  - unutilized credit lines

- Nature of borrowings:
  - commercial papers
  - short-term funding

- Purpose of liquidity:
  - margin requirements at exchange
  - funding MTF book

- Asset liquidity:
  - nature of MTF book (short-term, liquidatable)

- Parent support:
  - financial flexibility due to strong parentage

IMPORTANT:
- Use ONLY what is explicitly mentioned
- Do NOT infer or generalize
- Do NOT skip any numerical or qualitative detail

---

STEP 2: PARAGRAPH GENERATION (CRITICAL)

Write ONE concise paragraph that follows this flow:

1. Start with:
   → Overall liquidity support (e.g., parentage)

2. Then:
   → Nature of liquidity requirement (exchange margins, MTF book)

3. Then:
   → Borrowings structure (commercial papers, amount if mentioned)

4. Then:
   → Available liquidity sources:
      - cash & equivalents
      - credit lines
      - unencumbered cash

5. Then:
   → Asset-side liquidity (MTF book short-term nature)

6. End with:
   → Clear conclusion:
     "liquidity can be considered adequate"

---

MUST INCLUDE (if present in text):

- Borrowings amount and form (commercial papers)
- Cash & cash equivalent levels
- Unutilized credit lines
- Unencumbered cash
- Short-term nature of MTF book
- Parent support

---

WRITING STYLE (VERY IMPORTANT):

- Neutral, rating-agency tone
- Use phrases like:
  "financial flexibility is supported by..."
  "liquidity requirement is mainly for..."
  "borrowings are in the form of..."
  "cash and cash equivalents stood at..."
  "further, the MTF book is short-term in nature..."
  "hence, liquidity can be considered adequate"

- DO NOT:
  - Use words like "strong", "robust"
  - Over-explain
  - Add new interpretation
  - Skip any key liquidity component

---

STRICT RULE:

All explicitly mentioned liquidity components MUST be included.

Missing any of:
- borrowings
- cash
- credit lines
- MTF liquidity
→ makes output incorrect

---

OUTPUT FORMAT:

**Liquidity**

<Single analytical paragraph ending with conclusion>
"""

In [26]:
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

file = client.files.upload(
    file="/Users/aryamanchandak/Documents/ICICI Prudential/Code/Debt Dashboard/Debt Credit Eval Note/360oneWAM_AR.pdf"
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        file,
        prompt
    ]
)

Markdown(response.text)

**Liquidity**

The Group’s financial flexibility is supported by its well-defined Asset Liability Management (ALM) Framework, which includes maintaining sufficient cash, marketable securities, and access to adequate credit lines. The primary liquidity requirement is to meet short-term financial obligations. As of March 31, 2025, total financial liabilities stood at Rs. 12,389.75 crore, with short-term liabilities (due within one year) amounting to Rs. 9,158.11 crore. Borrowings are predominantly in the form of debt securities (Rs. 7,909.78 crore), which include secured bonds/debentures of Rs. 5,183.59 crore and unsecured commercial papers of Rs. 2,821.00 crore. Other borrowings (excluding debt securities) of Rs. 3,050.29 crore include collateralized borrowing and lending obligations (CBLO) of Rs. 2,120.00 crore (secured against investments in government securities), secured term loans from banks of Rs. 502.66 crore, and secured demand loans from banks of Rs. 352.64 crore (both backed by corporate guarantee). Subordinated liabilities were Rs. 134.67 crore. There have been no reported delays or defaults in repayment of principal or interest on debt securities and borrowings. Liquid funds comprise cash and cash equivalents of Rs. 740.19 crore and unencumbered bank balances of Rs. 7.59 crore as of March 31, 2025. Further, unutilized credit lines (undrawn backup lines) of Rs. 150.00 crore provide additional financial flexibility. On the asset side, loans of Rs. 8,397.37 crore are largely against securities/collaterals, and trade receivables of Rs. 443.06 crore are predominantly short-term in nature. Investments of Rs. 7,608.12 crore also contribute to asset liquidity. Hence, liquidity can be considered adequate.

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd

# Show all rows
pd.set_option("display.max_rows", None)

# Show all columns
pd.set_option("display.max_columns", None)

# Disable column width truncation"
pd.set_option("display.max_colwidth", None)

# Use full notebook width
pd.set_option("display.width", None)

In [ ]:
import requests
from datetime import datetime
import json
import pandas as pd
import pytz

def get_bse_announcements():

    url = "https://api.bseindia.com/BseIndiaAPI/api/AnnSubCategoryGetData/w"

    today_date = datetime.now().strftime("%Y%m%d")

    params = {
        'pageno': '1',
        'strCat': '-1',
        'strPrevDate': today_date,
        'strToDate': today_date,
        'strType': 'C',
        'strSearch': 'P',
        'strScrip': '',
        'subcategory': '-1'
    }

    headers = {
        'Referer': 'https://www.bseindia.com/',
        'User-Agent': 'Mozilla/5.0',
        'Accept': 'application/json, text/plain, */*'
    }

    try:
        print(f"Fetching data for date: {today_date}...")
        response = requests.get(url, params=params, headers=headers)

        if response.status_code == 200:
            data = response.json()

            # Save JSON
            with open("bse_announcements.json", "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4)

            print("JSON saved successfully.")

            # Create DataFrame
            if "Table" in data:
                df = pd.DataFrame(data["Table"])
                print("DataFrame created successfully.")

                # Optional: Save CSV
                df.to_csv("bse_announcements.csv", index=False)
                print("CSV file saved successfully.")

                return df
            else:
                print("No 'Table' key found in response.")

        else:
            print("Request failed:", response.status_code)

    except Exception as e:
        print(f"Exception: {e}")

# Run function
df2 = get_bse_announcements()

df2["SCRIP_CD"] = df2["SCRIP_CD"].astype("string")

df2["DissemDT"] = pd.to_datetime(
    df2["DissemDT"],
    format="%Y-%m-%dT%H:%M:%S.%f",
    errors="coerce"
).dt.tz_localize("Asia/Kolkata")

df2 = df2[['SCRIP_CD','SLONGNAME','CATEGORYNAME','SUBCATNAME','NEWSSUB','HEADLINE','MORE','DissemDT','ATTACHMENTNAME']]
df2['ATTACHMENTNAME'] = 'https://www.bseindia.com/xml-data/corpfiling/AttachLive/' + df2['ATTACHMENTNAME']

df2.shape
df2.head()

In [ ]:
You are a macroeconomic analyst preparing a monthly update on the Indian economy for an institutional audience.

Your task is to summarize the most relevant economic developments, policy actions, and structural trends for the month of {MONTH YEAR}.

The output should be informative, neutral in tone, and suitable for sharing with global stakeholders. Avoid opinions, recommendations, or strong conclusions. Focus on clarity, factual accuracy, and relevance.

Prioritize credible sources such as official government releases, established publications, and reputed global media. Where appropriate, attribute key data points.

Focus on:
- Major macroeconomic indicators and trends
- Government policy announcements and regulatory changes
- Trade, investment, and sector-level developments
- Any notable shifts in economic direction or priorities

Additionally, capture how international media or global observers are reacting to India’s economic developments during the month, where such coverage is meaningful.

Structure the output in a clear and readable manner. A suggested flow is:
1. Brief macro overview
2. Key developments (can be in bullet or paragraph form)
3. Any notable global perspective or commentary

Keep the write-up concise (approximately 250–400 words), avoiding unnecessary detail or repetition.

If certain areas lack sufficient credible information, prioritize accuracy over completeness and do not speculate.

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd

# Show all rows
pd.set_option("display.max_rows", None)

# Show all columns
pd.set_option("display.max_columns", None)

# Disable column width truncation"
pd.set_option("display.max_colwidth", None)

# Use full notebook width
pd.set_option("display.width", None)

In [ ]:
prompt = '''
You are a macroeconomic analyst working in the International Business team of an Indian Asset Management Company.

Your task is to prepare a concise monthly update on developments in the Indian economy for {MONTH YEAR}, intended for global institutional stakeholders.

Focus on capturing:
- Key macroeconomic trends
- Important policy and regulatory developments
- Structural or sector-level signals
- Noteworthy economic data points
- How global media is reacting to India

The objective is to inform, not to advise.

Maintain a neutral, factual, and professional tone throughout. Avoid strong opinions, investment recommendations, or commentary on individual companies unless absolutely necessary for context.

Prioritize credible and verifiable sources such as:
- Government releases (PIB, RBI, ministries, etc.)
- Established Indian publications
- Reputed global media (especially for international perception)

If data is uncertain or unavailable, avoid speculation.

Structure the output in a clear and readable format. A typical flow could include:
- A short opening overview of the macro environment
- A set of key developments or events from the month
- Additional context for a few important developments (if relevant)
- A brief view of global media or international sentiment toward India

Keep the write-up concise but informative, similar to an institutional monthly note. Ensure the content reflects only developments from March 2026, unless minimal context is required.

The final output should be suitable for direct sharing with global clients.
'''

In [ ]:
import time
from google import genai
from IPython.display import Markdown

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

interaction = client.interactions.create(
    input=prompt,
    agent='deep-research-pro-preview-12-2025',
    background=True
)

print(f"Research started: {interaction.id}")

while True:
    interaction = client.interactions.get(interaction.id)
    if interaction.status == "completed":
        Markdown(interaction.outputs[-1].text)
        break
    elif interaction.status == "failed":
        print(f"Research failed: {interaction.error}")
        break
    time.sleep(10)

In [ ]:
from IPython.display import Markdown

Markdown(interaction.outputs[-1].text)

In [ ]:
import time
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

interaction = client.interactions.create(
    input=prompt,
    agent='deep-research-pro-preview-12-2025',
    background=True
)

print(f"Research started: {interaction.id}")

while True:
    interaction = client.interactions.get(interaction.id)
    if interaction.status == "completed":
        print(interaction.outputs[-1].text)
        break
    elif interaction.status == "failed":
        print(f"Research failed: {interaction.error}")
        break
    time.sleep(10)

In [ ]:
1+2


In [ ]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

folder_path = "/content/docs"

uploaded_files = []

for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)

    if os.path.isfile(file_path):
        print("Uploading:", file_name)

        file = client.files.upload(
            file=file_path
        )

        uploaded_files.append(file)

print("Total uploaded:", len(uploaded_files))

In [ ]:
store = client.file_search_stores.create()

print("Store created:", store.name)

In [ ]:
import time

operations = []

for file in uploaded_files:

    print("Indexing:", file.display_name)

    op = client.file_search_stores.import_file(
        file_search_store_name=store.name,
        file_name=file.name,
        config={
          'chunking_config': {
            'white_space_config': {
              'max_tokens_per_chunk': 500,
              'max_overlap_tokens': 50
            }
          }
      }
    )

    operations.append(op)

print("Import started for", len(operations), "files")

In [ ]:
for op in operations:

    while not op.done:
        op = client.operations.get(op)

print("All files indexed successfully")

In [ ]:
MONTH = "March 2026"

prompt = f"""
You are a macroeconomic analyst working in the International Business team of an Indian Asset Management Company.

Prepare a monthly update on developments in the Indian economy for {MONTH}, intended for global institutional stakeholders.

Focus on:
- Key macroeconomic trends
- Policy and regulatory developments
- Important data points
- Structural or sector-level signals
- Global media reaction to India

Use both:
- Public web sources
- Provided internal PDF documents

Ensure a balanced use of internal documents and external web sources. Internal PDFs should be used wherever they provide relevant context or validation, alongside recent web information.

When using internal documents:
- Clearly mention the exact PDF name
- Mention the page number wherever possible
- Keep citations concise and naturally integrated into the sentence

Example:
“Fiscal trends remained stable (Source: Nomura_India_Outlook_Jan2026.pdf, Page 12)”

When using web sources:
- Prefer credible and established publications
- Attribute important data points where relevant

Maintain a neutral, factual, and professional tone. Avoid strong opinions, investment recommendations, or commentary on individual companies unless necessary for context.

Keep the output concise, well-structured, and suitable for client communication.
Ensure the content primarily reflects developments from {MONTH}.
"""

In [ ]:
interaction = client.interactions.create(
    input=prompt,
    agent="deep-research-pro-preview-12-2025",
    background=True,
    tools=[
        {
            "type": "file_search",
            "file_search_store_names": [store.name]
        }
    ]
)

print("Research started:", interaction.id)

In [ ]:
while True:
    result = client.interactions.get(interaction.id)

    if result.status == "completed":
        print("\n===== FINAL OUTPUT =====\n")
        print(result.outputs[-1].text)
        break

    elif result.status == "failed":
        print("Error:", result.error)
        break

    else:
        print("Still processing...")
        time.sleep(10)

In [ ]:
from IPython.display import Markdown

Markdown(result.outputs[-1].text)

In [ ]:
1+2